In [1]:
# Import core libraries for data manipulation
import os
import numpy as np
import pandas as pd

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
# Resolve dataset directory path safely
if os.path.exists('data/raw/oulad'):
    data_path = 'data/raw/oulad/'
elif os.path.exists('../data/raw/oulad'):
    data_path = '../data/raw/oulad/'
else:
    data_path = 'D:/STD_SUPPORT_project/data/raw/oulad/'

print(f"Loading raw datasets from: {data_path}")


Loading raw datasets from: ../data/raw/oulad/


In [3]:
# Load student demographic and registration tables
student_info = pd.read_csv(data_path + 'studentInfo.csv', na_values='?')
student_reg = pd.read_csv(data_path + 'studentRegistration.csv', na_values='?')

# Load assessment metadata and student submissions
assessments = pd.read_csv(data_path + 'assessments.csv', na_values='?')
student_assessment = pd.read_csv(data_path + 'studentAssessment.csv', na_values='?')

# Load student VLE clickstream with only required columns
vle_cols = ['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']
student_vle = pd.read_csv(data_path + 'studentVle.csv', usecols=vle_cols)

print("All datasets loaded successfully:")
print(f"- student_info: {student_info.shape}")
print(f"- student_reg: {student_reg.shape}")
print(f"- assessments: {assessments.shape}")
print(f"- student_assessment: {student_assessment.shape}")
print(f"- student_vle: {student_vle.shape}")


All datasets loaded successfully:
- student_info: (32593, 12)
- student_reg: (32593, 5)
- assessments: (206, 6)
- student_assessment: (173912, 5)
- student_vle: (10655280, 6)


In [4]:
# 1. Merge student_info with student_reg
merged_students = pd.merge(
    student_info,
    student_reg[['code_module', 'code_presentation', 'id_student', 'date_registration', 'date_unregistration']],
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

# 2. Identify students who unregistered on or before Day 30
is_early_unreg = (merged_students['date_unregistration'].notnull()) & (merged_students['date_unregistration'] <= 30)

# 3. Identify the 93 Withdrawn students with missing unregistration dates
is_withdrawn_unrecorded = (merged_students['final_result'] == 'Withdrawn') & (merged_students['date_unregistration'].isnull())

# 4. Filter to active Day 30 cohort
cohort = merged_students[~is_early_unreg & ~is_withdrawn_unrecorded].copy()

print(f"Total Eligible Cohort Rows: {len(cohort):,}")


Total Eligible Cohort Rows: 27,373


In [5]:
# Verify composite key uniqueness
composite_key_count = cohort.groupby(['code_module', 'code_presentation', 'id_student']).ngroups
print(f"Unique composite key combinations: {composite_key_count:,}")
print(f"Is every row a unique enrollment? {composite_key_count == len(cohort)}")


Unique composite key combinations: 27,373
Is every row a unique enrollment? True


In [6]:
# Preview demographic and academic background columns
demo_cols = [
    'code_module', 'code_presentation', 'id_student',
    'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability',
    'num_of_prev_attempts', 'studied_credits', 'date_registration'
]

cohort[demo_cols].head(5)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,disability,num_of_prev_attempts,studied_credits,date_registration
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,N,0,240,-159.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,N,0,60,-53.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,N,0,60,-52.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,N,0,60,-176.0
5,AAA,2013J,38053,M,Wales,A Level or Equivalent,80-90%,35-55,N,0,60,-110.0


In [7]:
# Summary statistics of numerical demographic and academic features
cohort[['num_of_prev_attempts', 'studied_credits', 'date_registration']].describe().round(2)


,num_of_prev_attempts,studied_credits,date_registration
count,27373.00,27373.00,27366.00
mean,0.16,76.26,-66.37
std,0.47,38.21,47.62
min,0.00,30.00,-311.00
25%,0.00,60.00,-95.00
50%,0.00,60.00,-53.00
75%,0.00,90.00,-29.00
max,6.00,630.00,167.00


In [8]:
# 1. Filter student_vle strictly to interactions on or before Day 30
vle_early = student_vle[student_vle['date'] <= 30]

print(f"Total VLE interaction rows on or before Day 30: {len(vle_early):,}")


Total VLE interaction rows on or before Day 30: 2,980,575


In [9]:
# 2. Aggregate primary early engagement metrics per enrollment
vle_agg = vle_early.groupby(['code_module', 'code_presentation', 'id_student']).agg(
    total_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique'),
    unique_sites=('id_site', 'nunique')
).reset_index()

vle_agg.head(5)


,code_module,code_presentation,id_student,total_clicks,active_days,unique_sites
0,AAA,2013J,11391,424,10,26
1,AAA,2013J,28400,618,19,34
2,AAA,2013J,30268,281,12,22
3,AAA,2013J,31604,540,24,32
4,AAA,2013J,32885,567,24,34


In [10]:
# 3. Aggregate pre-course engagement (clicks before Day 0)
vle_pre = student_vle[student_vle['date'] < 0].groupby(['code_module', 'code_presentation', 'id_student']).agg(
    pre_course_clicks=('sum_click', 'sum')
).reset_index()

# Combine VLE features into a single summary table
vle_features = pd.merge(vle_agg, vle_pre, on=['code_module', 'code_presentation', 'id_student'], how='left')

# If a student had no pre-course clicks, fill with 0
vle_features['pre_course_clicks'] = vle_features['pre_course_clicks'].fillna(0)

vle_features.head(5)


,code_module,code_presentation,id_student,total_clicks,active_days,unique_sites,pre_course_clicks
0,AAA,2013J,11391,424,10,26,98.0
1,AAA,2013J,28400,618,19,34,215.0
2,AAA,2013J,30268,281,12,22,102.0
3,AAA,2013J,31604,540,24,32,169.0
4,AAA,2013J,32885,567,24,34,295.0


In [11]:
# 1. Merge student assessment submissions with assessment metadata
sa_merged = pd.merge(student_assessment, assessments, on='id_assessment', how='left')

# 2. Keep only submissions completed on or before Day 30
sa_early = sa_merged[sa_merged['date_submitted'] <= 30]

print(f"Total assessment submissions on or before Day 30: {len(sa_early):,}")


Total assessment submissions on or before Day 30: 26,522


In [12]:
# 3. Aggregate early assessment metrics per student enrollment
sa_agg = sa_early.groupby(['code_module', 'code_presentation', 'id_student']).agg(
    early_assessments_submitted=('id_assessment', 'count'),
    early_avg_score=('score', 'mean')
).reset_index()

sa_agg.head(5)


,code_module,code_presentation,id_student,early_assessments_submitted,early_avg_score
0,AAA,2013J,11391,1,78.0
1,AAA,2013J,28400,1,70.0
2,AAA,2013J,31604,1,72.0
3,AAA,2013J,32885,1,69.0
4,AAA,2013J,38053,1,79.0


In [13]:
# 1. Merge cohort with VLE features
df_combined = pd.merge(
    cohort,
    vle_features,
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

# 2. Merge with early assessment features
df_combined = pd.merge(
    df_combined,
    sa_agg,
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

print(f"Combined DataFrame shape: {df_combined.shape}")


Combined DataFrame shape: (27373, 20)


In [14]:
# 3. Impute natural zero counts for engagement and submissions
zero_fill_cols = [
    'total_clicks', 
    'active_days', 
    'unique_sites', 
    'pre_course_clicks', 
    'early_assessments_submitted'
]

for col in zero_fill_cols:
    df_combined[col] = df_combined[col].fillna(0)

# 4. Create binary indicator for early submission
df_combined['has_early_submission'] = (df_combined['early_assessments_submitted'] > 0).astype(int)

print("Zero-activity counts imputed successfully!")
df_combined[['total_clicks', 'active_days', 'early_assessments_submitted', 'has_early_submission']].head(5)


Zero-activity counts imputed successfully!


,total_clicks,active_days,early_assessments_submitted,has_early_submission
0,424.0,10.0,1.0,1
1,618.0,19.0,1.0,1
2,540.0,24.0,1.0,1
3,567.0,24.0,1.0,1
4,730.0,33.0,1.0,1


In [15]:
# 1. Create binary target column
# 0 = Not At Risk (Pass or Distinction)
# 1 = At Risk (Fail or Withdrawn after Day 30)
df_combined['target'] = df_combined['final_result'].apply(
    lambda x: 1 if x in ['Fail', 'Withdrawn'] else 0
)

# 2. Display target distribution
target_dist = pd.DataFrame({
    'Student Count': df_combined['target'].value_counts(),
    'Percentage (%)': (df_combined['target'].value_counts(normalize=True) * 100).round(2)
})
target_dist


,Student Count,Percentage (%)
target,,
0,15385,56.21
1,11988,43.79


In [16]:
# 3. Drop post-cutoff outcome columns to prevent data leakage
columns_to_drop = ['final_result', 'date_unregistration']
df_final = df_combined.drop(columns=columns_to_drop)

# Reorder columns logically: Identifiers -> Demographics -> Engagement -> Assessments -> Target
final_column_order = [
    'code_module', 'code_presentation', 'id_student',
    'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability',
    'num_of_prev_attempts', 'studied_credits', 'date_registration',
    'total_clicks', 'active_days', 'unique_sites', 'pre_course_clicks',
    'early_assessments_submitted', 'early_avg_score', 'has_early_submission',
    'target'
]

df_final = df_final[final_column_order]

print(f"Final modeling dataset shape: {df_final.shape}")
df_final.head(3)


Final modeling dataset shape: (27373, 20)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,disability,num_of_prev_attempts,studied_credits,date_registration,total_clicks,active_days,unique_sites,pre_course_clicks,early_assessments_submitted,early_avg_score,has_early_submission,target
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,N,0,240,-159.0,424.0,10.0,26.0,98.0,1.0,78.0,1,0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,N,0,60,-53.0,618.0,19.0,34.0,215.0,1.0,70.0,1,0
2,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,N,0,60,-52.0,540.0,24.0,32.0,169.0,1.0,72.0,1,0


In [17]:
# 1. Check row and column dimensions
print("=== 1. Dataset Dimensions ===")
print(f"Total Rows: {df_final.shape[0]:,}")
print(f"Total Columns: {df_final.shape[1]}")


=== 1. Dataset Dimensions ===
Total Rows: 27,373
Total Columns: 20


In [18]:
# 2. Check composite key uniqueness
print("=== 2. Composite Key Uniqueness ===")
unique_keys = df_final.groupby(['code_module', 'code_presentation', 'id_student']).ngroups
print(f"Unique composite key combinations: {unique_keys:,}")
print(f"Is composite key 100% unique? {unique_keys == len(df_final)}")


=== 2. Composite Key Uniqueness ===
Unique composite key combinations: 27,373
Is composite key 100% unique? True


In [19]:
# 3. Check duplicate rows
print("=== 3. Duplicate Rows ===")
dup_count = df_final.duplicated().sum()
print(f"Exact duplicate rows: {dup_count}")


=== 3. Duplicate Rows ===
Exact duplicate rows: 0


In [20]:
# 4. Check missing values
print("=== 4. Missing Values Audit ===")
null_series = df_final.isnull().sum()
null_df = pd.DataFrame({
    'Missing Count': null_series,
    'Missing (%)': (null_series / len(df_final) * 100).round(2)
})
null_df = null_df[null_df['Missing Count'] > 0]
null_df


=== 4. Missing Values Audit ===


,Missing Count,Missing (%)
imd_band,1026,3.75
date_registration,7,0.03
early_avg_score,6546,23.91


In [21]:
# 5. Check target distribution
print("=== 5. Target Distribution ===")
target_counts = df_final['target'].value_counts()
target_percents = (df_final['target'].value_counts(normalize=True) * 100).round(2)

pd.DataFrame({
    'Class': [0, 1],
    'Description': ['Not At Risk (Pass / Distinction)', 'At Risk (Fail / Post-Day 30 Withdrawn)'],
    'Count': [target_counts[0], target_counts[1]],
    'Percentage (%)': [target_percents[0], target_percents[1]]
})


=== 5. Target Distribution ===


,Class,Description,Count,Percentage (%)
0,0,Not At Risk (Pass / Distinction),15385,56.21
1,1,At Risk (Fail / Post-Day 30 Withdrawn),11988,43.79


In [22]:
# Ensure output directory exists
if os.path.exists('data/processed'):
    output_dir = 'data/processed'
elif os.path.exists('../data/processed'):
    output_dir = '../data/processed'
else:
    output_dir = 'D:/STD_SUPPORT_project/data/processed'

os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'student_model_data.csv')

# Save DataFrame to CSV without index
df_final.to_csv(output_path, index=False)

# Verify file existence and file size
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Successfully saved modeling dataset to: {output_path}")
print(f"File size: {file_size_mb:.2f} MB")


Successfully saved modeling dataset to: ../data/processed\student_model_data.csv
File size: 2.96 MB


In [23]:
# Reload the saved CSV to verify integrity
df_check = pd.read_csv(output_path)
print(f"Reload verification: {df_check.shape[0]:,} rows, {df_check.shape[1]} columns.")


Reload verification: 27,373 rows, 20 columns.
